E-COMMERCE PROJECT — STEP 1: CREATE DATASET

In [5]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta  # works with dates

 STEP 2: SET A SEED 

In [6]:
random.seed(42)
np.random.seed(42)


# DEFINE THE BUILDING BLOCKS OF OUR DATASET

In [ ]:
n_customers = 500   # i created 500 unique customers
# Cities where customers are located
cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai','Hyderabad', 'Pune', 'Kolkata', 'Ahmedabad']
# Payment methods used
payment_methods = ['UPI', 'Credit Card', 'Debit Card','COD', 'Net Banking']


In [9]:
products_info = {
    'iPhone 15':          ('Electronics', 70000, 90000),
    'Samsung Galaxy S23': ('Electronics', 40000, 60000),
    'Laptop Stand':       ('Electronics',  1500,  3500),
    'Wireless Earbuds':   ('Electronics',  1200,  8000),
    'Kurta Set':          ('Fashion',       600,  3500),
    'Running Shoes':      ('Fashion',      1800,  7000),
    'Formal Shirt':       ('Fashion',       900,  2800),
    'Python Book':        ('Books',         350,   900),
    'Data Science Book':  ('Books',         450,  1300),
    'Face Serum':         ('Beauty',        500,  2800),
    'Moisturizer':        ('Beauty',        250,  1600),
    'Perfume':            ('Beauty',        900,  5500),
    'Bed Sheet Set':      ('Home',          900,  3200),
    'Air Purifier':       ('Home',         5500, 16000),
    'Cookware Set':       ('Home',         2200,  9000),
}

In [10]:
product_names = list(products_info.keys())  # list of product names only

In [11]:
# CUSTOMER BEHAVIOUR TYPES
# What % of customers are each type?
cust_types   = ['champion', 'loyal', 'at_risk', 'new', 'lost']
#these percentages were chosen intentionally to create a realistic-looking sample dataset.
cust_weights = [0.15,       0.25,    0.20,      0.20,  0.20]

In [13]:
# How many orders each type places (min, max range)
order_counts = {
    'champion': (10, 22),   # Champions buy a LOT
    'loyal':    ( 5,  9),   # Loyal customers buy regularly
    'at_risk':  ( 3,  6),   # Used to buy, now slowing down
    'new':      ( 1,  3),   # Just joined, few purchases
    'lost':     ( 1,  2),   # Bought once and disappeared
}


In [15]:
# How recently (days ago) did they last purchase?
#Recency means: How recently a customer made their last purchase.
recency_ranges = {
    'champion': (  1,  25),   # Bought very recently #its been 1-25 days since their last purchase.
    'loyal':    ( 15,  55),   # Bought recently, but not as recent as champions #its been 15-55 days since their last purchase.
    'at_risk':  ( 60, 150),   # Hasn't bought in 2-5 months!
    'new':      (  5,  40),   # Just joined, so bought recently, but not as recent as champions #its been 5-40 days since their last purchase.
    'lost':     (180, 365),   # Hasn't bought in 6-12 months
}

In [17]:
# How much do they spend per order (multiplier on base price)
spend_mult = {
    'champion': 2.0,   # Spends 2x average
    'loyal':    1.3,
    'at_risk':  1.0,
    'new':      0.8,
    'lost':     0.6,   # Only ever bought cheap things
}
# champion: this type of customer is the most valuable. They buy frequently, spend a lot, and have made recent purchases. They are the "champions" of your business.
# loyal:This customer buys regularly and spends a bit more than average.
# at_risk:This customer spends about the average amount.
# new:This customer is trying the store for the first time and makes a small purchase.
# lost:This customer used to buy only low-cost items and eventually stopped purchasing.

main dataset generation loop.
"Create customers → assign a customer type → generate their orders → store everything in a dataset."



In [19]:
# GENERATE ALL ORDERS
orders = []      
today  = datetime(2024, 1, 1) 
for cust_num in range(1, n_customers + 1):
    cust_id   = f'C{cust_num:04d}'
    city      = random.choice(cities)
    cust_type = random.choices(cust_types, weights=cust_weights)[0]
    min_o, max_o = order_counts[cust_type]
    num_orders   = random.randint(min_o, max_o)
    min_d, max_d = recency_ranges[cust_type]
    days_ago     = random.randint(min_d, max_d)
    last_date    = today - timedelta(days=days_ago)
    for i in range(num_orders):
        spread_days  = random.randint(0, 300)
        order_date   = last_date - timedelta(days=spread_days * i / max(num_orders, 1))
        order_date = max(order_date, datetime(2023, 1, 1))
        order_date = min(order_date, datetime(2023, 12, 31))
        product = random.choice(product_names)
        cat, min_p, max_p = products_info[product]
        unit_price = random.uniform(min_p, max_p) * spend_mult[cust_type]
        quantity   = random.randint(1, 4)
        total      = round(unit_price * quantity, 2)
        returned   = random.random() < 0.05  # True 5% of the time
        orders.append({
            'customer_id':    cust_id,
            'order_id':       f'ORD{len(orders)+1:05d}',
            'order_date':     order_date.strftime('%Y-%m-%d'),
            'product_name':   product,
            'category':       cat,
            'quantity':       quantity,
            'unit_price':     round(unit_price, 2),
            'total_amount':   total,
            'payment_method': random.choice(payment_methods),
            'city':           city,
            'returned':       returned,
            'true_segment':   cust_type,  # for validation only
        })
df = pd.DataFrame(orders)
df.to_csv('ecommerce_orders.csv', index=False)
print(f"✅ Dataset created!")


✅ Dataset created!


In [20]:
print(f"Rows (orders):{len(df)}")

Rows (orders):3043


In [21]:
print(f"   Unique customers:    {df['customer_id'].nunique()}")
print(f"   Total Revenue (₹):   {df['total_amount'].sum():,.0f}")
print(f"   Date range:          {df['order_date'].min()} to {df['order_date'].max()}")

   Unique customers:    500
   Total Revenue (₹):   127,227,726
   Date range:          2023-01-01 to 2023-12-31


In [22]:
print()

In [23]:
print(df.head(10))

  customer_id  order_id  order_date        product_name     category  \
0       C0001  ORD00001  2023-12-24        Laptop Stand  Electronics   
1       C0001  ORD00002  2023-12-22  Samsung Galaxy S23  Electronics   
2       C0001  ORD00003  2023-11-23    Wireless Earbuds  Electronics   
3       C0001  ORD00004  2023-12-06             Perfume       Beauty   
4       C0001  ORD00005  2023-12-09  Samsung Galaxy S23  Electronics   
5       C0001  ORD00006  2023-11-05       Bed Sheet Set         Home   
6       C0001  ORD00007  2023-12-06   Data Science Book        Books   
7       C0001  ORD00008  2023-12-12         Moisturizer       Beauty   
8       C0001  ORD00009  2023-11-24        Formal Shirt      Fashion   
9       C0001  ORD00010  2023-10-16         Moisturizer       Beauty   

   quantity  unit_price  total_amount payment_method   city  returned  \
0         1     5945.88       5945.88            UPI  Delhi     False   
1         1    88745.52      88745.52    Net Banking  Delhi  